In [1]:
import pandas as pd
import os
import numpy as np

from pathlib import Path

ROOT_DIR = Path(os.getcwd()).parent

In [2]:
df = pd.read_parquet(os.path.join(ROOT_DIR, "data/data_transformed.parquet"))
df = df.reset_index(drop=True)
df.shape

(5, 11425)

In [6]:
df

,chi2_stat_max_100_H,chi2_stat_median_100_H,hour_bin_0-3_max_100_H,hour_bin_0-3_median_100_H,hour_bin_12-15_max_100_H,hour_bin_12-15_median_100_H,hour_bin_15-18_max_100_H,hour_bin_15-18_median_100_H,hour_bin_18-21_max_100_H,hour_bin_18-21_median_100_H,...,weekday_4_max_9_H,weekday_4_median_9_H,weekday_5_max_9_H,weekday_5_median_9_H,weekday_6_max_9_H,weekday_6_median_9_H,weekday_7_max_9_H,weekday_7_median_9_H,whale_imbalance_ratio_mean_9_H,is_pumped
0,32.759946,31.097093,0.064240,0.062057,0.267715,0.236539,0.066188,0.063904,0.275128,0.260039,...,2.810079e-01,2.743057e-01,4.357190e-02,4.253269e-02,0.181986,0.177646,0.194093,0.189464,-0.018996,False
1,32.759946,31.097093,0.064240,0.062057,0.267715,0.236539,0.066188,0.063904,0.275128,0.260039,...,2.810079e-01,2.743057e-01,4.357190e-02,4.253269e-02,0.181986,0.177646,0.194093,0.189464,-0.018996,False
2,32.759946,31.097093,0.064240,0.062057,0.267715,0.236539,0.066188,0.063904,0.275128,0.260039,...,2.810079e-01,2.743057e-01,4.357190e-02,4.253269e-02,0.181986,0.177646,0.194093,0.189464,-0.018996,False
3,14.762604,11.301865,0.013733,0.013384,0.009011,0.008782,0.951014,0.948107,0.022102,0.021541,...,4.806365e-16,3.768545e-16,-9.034334e-18,-1.884273e-16,0.974070,0.459183,0.000000,0.000000,0.327277,False
4,3.119909,3.119909,0.152531,0.152531,0.133660,0.133660,0.258404,0.258404,0.232736,0.141396,...,1.411381e-01,1.411381e-01,5.114449e-01,5.114449e-01,0.347417,0.347417,0.000000,0.000000,-0.636283,False


In [ ]:
df["is_pumped"].value_counts()

In [3]:
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from functools import partial
import xgboost as xgb

df = df.fillna(0)

df_train, df_test = train_test_split(df, train_size=0.7, stratify=df["is_pumped"], shuffle=True)
df_val, df_test = train_test_split(df_test, train_size=0.7, stratify=df_test["is_pumped"], shuffle=True)

(
    df_train["is_pumped"].value_counts(), 
    df_test["is_pumped"].value_counts(),
    df_val["is_pumped"].value_counts()
)

NameError: name 'df' is not defined

In [ ]:
def xgboost_objective(trial, dtrain: xgb.DMatrix, dval: xgb.DMatrix) -> float:

    params = {
        "objective": "binary:logistic",
        "learning_rate": trial.suggest_float("learning_rate", 0.1, 1, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 100),
        "gamma": trial.suggest_float("gamma", 0.01, 1),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 500),
        "eval_metric": ["logloss"],
        "device": "cuda:0",
        "tree_method": "hist",
    }


    evals_result = {}

    model = xgb.train(
        params=params,
        dtrain=dtrain,
        evals=[(dtrain, "train"), (dval, "val")],
        num_boost_round=10000, early_stopping_rounds=10,
        verbose_eval=False, evals_result=evals_result,
    )

    y_pred = model.predict(dval)
    y_pred = (y_pred > trial.suggest_float("threshold", 0.5, 1, log=True)).astype(int)

    f1 = f1_score(y_true=dval.get_label(), y_pred=y_pred, pos_label=1)

    return f1

In [ ]:
dtrain = xgb.DMatrix(data=df_train.iloc[:, :-3], label=df_train["is_pumped"].astype(int))
dval = xgb.DMatrix(data=df_val.iloc[:, :-3], label=df_val["is_pumped"].astype(int))
dtest = xgb.DMatrix(data=df_test.iloc[:, :-3], label=df_test["is_pumped"].astype(int))

In [ ]:
import optuna

# study_xgboost = optuna.create_study(
#     direction="maximize",
# )

study_xgboost.optimize(
    partial(xgboost_objective, dtrain=dtrain, dval=dval),
    n_trials=100
)

In [ ]:
from optuna.visualization import plot_param_importances

plot_param_importances(study_xgboost)

In [ ]:
xgboost_params = {
    "objective": "binary:logistic",
    "eval_metric": ["logloss"],
    "device": "cuda:0",
    "tree_method": "hist",
}

xgboost_params.update(
    study_xgboost.best_params
)

In [ ]:
xgboost_model = xgb.train(
    params=xgboost_params,
    dtrain=dtrain,
    evals=[(dtrain, "train"), (dval, "val")],
    num_boost_round=5000, early_stopping_rounds=10,
    verbose_eval=True
)

In [ ]:
df_test["is_pumped"].value_counts()

In [ ]:
study_xgboost.best_params

In [ ]:
pred = xgboost_model.predict(
    dtest
)

y_pred = (pred > study_xgboost.best_params["threshold"]).astype(int)

In [ ]:
import numpy as np

np.unique(y_pred, return_counts=True)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

In [ ]:
cm = confusion_matrix(y_pred=y_pred, y_true=df_test["is_pumped"].astype(int))

ConfusionMatrixDisplay(cm).plot()